# 04 - Quantitative relationship analysis
Measures how the components relate, using the historical Olist data.

1. Customer and sales
2. Product and sales
3. Seller and sales
4. Sales and payment
5. Sales and delivery
6. Delivery and customer experience

All results are descriptive. Association is not causation.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# works whether the notebook is run from the project root or from notebooks/
here = Path.cwd().resolve()
PROJECT_ROOT = here.parent if here.name.lower() == "notebooks" else here
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "olist"

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Data folder not found: {DATA_DIR}. Run notebook 01 first.")

FILES = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}
dfs = {name: pd.read_csv(DATA_DIR / f) for name, f in FILES.items()}

for name, df in dfs.items():
    print(f"{name:22} rows={len(df):>9,}  cols={df.shape[1]}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

## 1. Prepare the tables
Each table is kept at its natural grain so joins do not multiply rows:
- `orders_analysis`: one row per order
- `item_analysis`: one row per order item
- payments and reviews are summed up to one row per order before joining

In [ ]:
customers = dfs["customers"].copy()
orders = dfs["orders"].copy()
items = dfs["order_items"].copy()
payments = dfs["payments"].copy()
reviews = dfs["reviews"].copy()
products = dfs["products"].copy()
sellers = dfs["sellers"].copy()
translation = dfs["category_translation"].copy()

# turn text timestamps into datetimes (bad values become NaT)
for col in ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
            "order_delivered_customer_date", "order_estimated_delivery_date"]:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

# add the real customer id (customer_unique_id) to each order
orders_analysis = orders.merge(
    customers[["customer_id", "customer_unique_id", "customer_city", "customer_state"]],
    on="customer_id", how="left", validate="many_to_one")

# order items with product category and seller location
item_analysis = (
    items
    .merge(products[["product_id", "product_category_name"]], on="product_id", how="left", validate="many_to_one")
    .merge(sellers[["seller_id", "seller_city", "seller_state"]], on="seller_id", how="left", validate="many_to_one")
    .merge(translation, on="product_category_name", how="left", validate="many_to_one")
)
# use the english name, fall back to the portuguese one, then "unknown"
item_analysis["category_english"] = (
    item_analysis["product_category_name_english"]
    .fillna(item_analysis["product_category_name"])
    .fillna("unknown")
)

# one row per order for items, payments and reviews
items_by_order = item_analysis.groupby("order_id", as_index=False).agg(
    item_count=("order_item_id", "count"),
    merchandise_value=("price", "sum"),
    freight_value=("freight_value", "sum"),
)
payment_by_order = payments.groupby("order_id", as_index=False).agg(
    payment_total=("payment_value", "sum"),
)
review_by_order = reviews.groupby("order_id", as_index=False).agg(
    mean_review_score=("review_score", "mean"),
)

orders_analysis = (
    orders_analysis
    .merge(items_by_order, on="order_id", how="left", validate="one_to_one")
    .merge(payment_by_order, on="order_id", how="left", validate="one_to_one")
    .merge(review_by_order, on="order_id", how="left", validate="one_to_one")
)

# delivery measures, in days
day = 86400
orders_analysis["delivery_days"] = (
    orders_analysis["order_delivered_customer_date"] - orders_analysis["order_purchase_timestamp"]
).dt.total_seconds() / day
# negative = delivered before the estimated date, positive = late
orders_analysis["delivery_vs_estimate_days"] = (
    orders_analysis["order_delivered_customer_date"] - orders_analysis["order_estimated_delivery_date"]
).dt.total_seconds() / day

print("orders_analysis:", orders_analysis.shape)
print("item_analysis:  ", item_analysis.shape)

### Quick check that the joins did not duplicate anything

In [ ]:
checks = pd.DataFrame([
    {"table": "orders_analysis", "rows": len(orders_analysis), "distinct_orders": orders_analysis["order_id"].nunique()},
    {"table": "payment_by_order", "rows": len(payment_by_order), "distinct_orders": payment_by_order["order_id"].nunique()},
    {"table": "review_by_order", "rows": len(review_by_order), "distinct_orders": review_by_order["order_id"].nunique()},
])
display(checks)

# how many orders are missing each measure
display(orders_analysis[["customer_unique_id", "item_count", "payment_total", "mean_review_score",
                         "delivery_days"]].isna().sum().to_frame("missing"))

## 2. Customer and sales
Repeat purchasing is counted per `customer_unique_id`, not per `customer_id`.

In [ ]:
customer_orders = (
    orders_analysis.dropna(subset=["customer_unique_id"])
    .groupby("customer_unique_id", as_index=False)
    .agg(order_count=("order_id", "nunique"),
         merchandise_spend=("merchandise_value", "sum"),
         freight_spend=("freight_value", "sum"),
         first_purchase=("order_purchase_timestamp", "min"),
         last_purchase=("order_purchase_timestamp", "max"))
)
customer_orders["repeat_customer"] = customer_orders["order_count"] > 1

summary = pd.DataFrame({
    "metric": ["Customers", "Ordered once", "Ordered more than once", "Repeat-customer share",
               "Mean orders per customer", "Median spend per customer"],
    "value": [len(customer_orders),
              int((customer_orders["order_count"] == 1).sum()),
              int(customer_orders["repeat_customer"].sum()),
              customer_orders["repeat_customer"].mean(),
              customer_orders["order_count"].mean(),
              customer_orders["merchandise_spend"].median()],
})
display(summary)
display(customer_orders.sort_values("order_count", ascending=False).head(10))

In [ ]:
counts = customer_orders["order_count"].value_counts().sort_index()

plt.figure(figsize=(9, 4))
plt.bar(counts.index.astype(str), counts.values)
plt.title("Orders per customer")
plt.xlabel("Number of orders")
plt.ylabel("Customers")
plt.tight_layout()
plt.show()

## 3. Product and sales
Value comes from `price` only. Freight is kept separate.

In [ ]:
product_performance = (
    item_analysis.groupby(["product_id", "category_english"], as_index=False)
    .agg(item_lines=("order_item_id", "count"),
         distinct_orders=("order_id", "nunique"),
         merchandise_value=("price", "sum"),
         freight_value=("freight_value", "sum"))
    .sort_values("merchandise_value", ascending=False)
)

category_performance = (
    item_analysis.groupby("category_english", as_index=False)
    .agg(item_lines=("order_item_id", "count"),
         distinct_orders=("order_id", "nunique"),
         distinct_products=("product_id", "nunique"),
         merchandise_value=("price", "sum"),
         freight_value=("freight_value", "sum"))
    .sort_values("merchandise_value", ascending=False)
)
category_performance["share"] = category_performance["merchandise_value"] / category_performance["merchandise_value"].sum()

print("Top 15 categories")
display(category_performance.head(15))
print("Top 15 products")
display(product_performance.head(15))

In [ ]:
top = category_performance.head(12).sort_values("merchandise_value")

plt.figure(figsize=(10, 6))
plt.barh(top["category_english"], top["merchandise_value"])
plt.title("Top categories by merchandise value")
plt.xlabel("Merchandise value")
plt.tight_layout()
plt.show()

## 4. Seller and sales
Sales contribution only, not profit or payout.

In [ ]:
seller_performance = (
    item_analysis.groupby("seller_id", as_index=False)
    .agg(item_lines=("order_item_id", "count"),
         distinct_orders=("order_id", "nunique"),
         distinct_products=("product_id", "nunique"),
         merchandise_value=("price", "sum"),
         freight_value=("freight_value", "sum"),
         seller_state=("seller_state", "first"))
    .sort_values("merchandise_value", ascending=False)
)
seller_performance["share"] = seller_performance["merchandise_value"] / seller_performance["merchandise_value"].sum()

print(f"Sellers with sales: {len(seller_performance):,}")
print(f"Top 10 sellers' share of merchandise value: {seller_performance.head(10)['share'].sum():.2%}")
display(seller_performance.head(15))

In [ ]:
top = seller_performance.head(12).sort_values("merchandise_value")

plt.figure(figsize=(10, 6))
plt.barh(top["seller_id"].str[:10], top["merchandise_value"])
plt.title("Top 12 sellers by merchandise value")
plt.xlabel("Merchandise value")
plt.ylabel("Seller id (first 10 characters)")
plt.tight_layout()
plt.show()

## 5. Sales and payment
A descriptive comparison, not an accounting reconciliation. Payment totals can include installments and several payment records per order.

In [ ]:
pay = orders_analysis.dropna(subset=["merchandise_value", "payment_total"]).copy()
pay["payment_minus_merchandise"] = pay["payment_total"] - pay["merchandise_value"]

payment_summary = pd.DataFrame({
    "metric": ["Orders compared", "Total merchandise value", "Total freight value", "Total payment value",
               "Median payment minus merchandise", "Orders paying more than merchandise",
               "Orders paying less than merchandise"],
    "value": [len(pay), pay["merchandise_value"].sum(), pay["freight_value"].sum(), pay["payment_total"].sum(),
              pay["payment_minus_merchandise"].median(),
              int((pay["payment_minus_merchandise"] > 0).sum()),
              int((pay["payment_minus_merchandise"] < 0).sum())],
})
display(payment_summary)

payment_method_summary = (
    payments.groupby("payment_type", as_index=False)
    .agg(records=("payment_sequential", "count"),
         distinct_orders=("order_id", "nunique"),
         payment_value=("payment_value", "sum"))
    .sort_values("payment_value", ascending=False)
)
display(payment_method_summary)

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(pay["merchandise_value"], pay["payment_total"], alpha=0.25)
plt.title("Merchandise value vs payment value (per order)")
plt.xlabel("Merchandise value")
plt.ylabel("Payment value")
plt.tight_layout()
plt.show()

## 6. Sales and delivery
`delivery_vs_estimate_days`: negative means delivered before the estimate, positive means late.

In [ ]:
delivered = orders_analysis.dropna(subset=["order_delivered_customer_date", "order_estimated_delivery_date"]).copy()
delivered["performance"] = np.select(
    [delivered["delivery_vs_estimate_days"] < 0, delivered["delivery_vs_estimate_days"] == 0],
    ["Early", "On estimated date"], default="Late")

delivery_summary = (
    delivered.groupby("performance", as_index=False)
    .agg(orders=("order_id", "nunique"),
         mean_days_vs_estimate=("delivery_vs_estimate_days", "mean"),
         mean_delivery_days=("delivery_days", "mean"))
)
delivery_summary["share"] = delivery_summary["orders"] / delivery_summary["orders"].sum()
display(delivery_summary)

# delivery timestamps only exist for some statuses
delivery_by_status = (
    orders_analysis.groupby("order_status", as_index=False)
    .agg(orders=("order_id", "nunique"),
         with_delivery_date=("order_delivered_customer_date", lambda s: s.notna().sum()),
         mean_delivery_days=("delivery_days", "mean"))
)
display(delivery_by_status)

In [ ]:
plot = delivery_summary.sort_values("orders")

plt.figure(figsize=(8, 4))
plt.bar(plot["performance"], plot["orders"])
plt.title("Delivered orders: actual vs estimated date")
plt.ylabel("Orders")
plt.tight_layout()
plt.show()

## 7. Delivery and customer experience
Reviews are averaged per order first. This is an association only: review scores also depend on product quality, seller service and more.

In [ ]:
experience = orders_analysis.dropna(subset=["mean_review_score", "delivery_vs_estimate_days"]).copy()
experience["performance"] = np.select(
    [experience["delivery_vs_estimate_days"] < 0, experience["delivery_vs_estimate_days"] == 0],
    ["Early", "On estimated date"], default="Late")

experience_summary = (
    experience.groupby("performance", as_index=False)
    .agg(orders=("order_id", "nunique"),
         mean_review_score=("mean_review_score", "mean"),
         median_review_score=("mean_review_score", "median"),
         mean_days_vs_estimate=("delivery_vs_estimate_days", "mean"))
    .sort_values("mean_days_vs_estimate")
)
display(experience_summary)

corr = experience["delivery_vs_estimate_days"].corr(experience["mean_review_score"])
print(f"Correlation (days vs estimate, review score): {corr:.3f} on {len(experience):,} orders")

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(experience["delivery_vs_estimate_days"], experience["mean_review_score"], alpha=0.2)
plt.title("Delivery vs estimate and review score")
plt.xlabel("Actual minus estimated delivery (days)")
plt.ylabel("Mean review score")
plt.tight_layout()
plt.show()

## 8. What each analysis can support

In [ ]:
relationship_evidence = pd.DataFrame([
    ("Customer - Order & Sales", "Orders and spend per customer", "Historical repeat behaviour is not a future purchase probability."),
    ("Product - Order & Sales", "Item lines and value by product and category", "No inventory or stock-out data."),
    ("Seller - Order & Sales", "Item lines and value by seller", "Not seller profit, payout or service quality."),
    ("Order & Sales - Payment", "Merchandise, freight and payment totals", "Not a full accounting reconciliation."),
    ("Order & Sales - Delivery", "Delivery time and actual vs estimate", "Timestamps do not explain why a delay happened."),
    ("Delivery - Customer experience", "Review score by delivery group", "Association, not causation."),
], columns=["Relationship", "What was measured", "Caution"])
display(relationship_evidence)

## 9. Save the summary tables
Written to `data/processed/relationship_analysis/`. The raw files are untouched.

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "relationship_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

outputs = {
    "customer_summary.csv": customer_orders,
    "product_performance.csv": product_performance,
    "category_performance.csv": category_performance,
    "seller_performance.csv": seller_performance,
    "payment_method_summary.csv": payment_method_summary,
    "delivery_summary.csv": delivery_summary,
    "delivery_by_order_status.csv": delivery_by_status,
    "delivery_review_summary.csv": experience_summary,
    "relationship_evidence.csv": relationship_evidence,
}
for filename, frame in outputs.items():
    frame.to_csv(OUTPUT_DIR / filename, index=False)

print(f"Saved {len(outputs)} tables to {OUTPUT_DIR}")